In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from sqlalchemy import create_engine
import pandas as pd

DB_PATH = '/content/drive/MyDrive/olist_project/olist.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

In [4]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", engine)
tables

,name
0,customers
1,orders
2,order_items
3,payments
4,reviews
5,products
6,sellers
7,geolocation
8,category_translation


In [22]:
orders = pd.read_sql("SELECT * FROM orders", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
payments = pd.read_sql("SELECT * FROM payments", engine)
reviews = pd.read_sql("SELECT * FROM reviews", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)
category_translation = pd.read_sql("SELECT * FROM category_translation", engine)

In [23]:
print("Orders shape:", orders.shape)
print("Null counts in orders:\n", orders.isnull().sum())
print("\nOrder status breakdown:\n", orders['order_status'].value_counts())

Orders shape: (99441, 8)

Null counts in orders:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status breakdown:
 order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


Number of rows in each table

In [5]:
tables = ['customers', 'orders', 'order_items', 'payments', 'reviews', 'products', 'sellers', 'geolocation', 'category_translation']
for t in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {t}", engine)
    print(f"{t}: {count['cnt'][0]} rows")

customers: 99441 rows
orders: 99441 rows
order_items: 112650 rows
payments: 103886 rows
reviews: 99224 rows
products: 32951 rows
sellers: 3095 rows
geolocation: 1000163 rows
category_translation: 71 rows


Date range of the orders table

In [6]:
pd.read_sql_query("""SELECT MIN(order_purchase_timestamp) AS earliest_order,
                      MAX(order_purchase_timestamp) AS latest_order
                      FROM orders""", engine)

,earliest_order,latest_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


3. Order status

In [7]:
pd.read_sql_query("""
    SELECT order_status, COUNT(*) AS num_orders
    FROM orders
    GROUP BY order_status
    ORDER BY num_orders DESC
""", engine)

,order_status,num_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [13]:
pd.read_sql_query("""select * from reviews limit 5""",engine)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


Review Score Distribution

In [12]:
pd.read_sql_query("""SELECT review_score, count(*) AS num_reviews FROM reviews GROUP BY review_score""",engine)

,review_score,num_reviews
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


In [14]:
pd.read_sql_query("""select * from payments limit 5""",engine)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Basic payment check

In [19]:
pd.read_sql_query("""SELECT MIN(payment_value) as min_payment, MAX(payment_value) as max_payment,
                    AVG(payment_value) as avg_payment FROM payments""",engine)

,min_payment,max_payment,avg_payment
0,0.0,13664.08,154.10038


In [21]:
orders = pd.read_sql("SELECT * FROM orders", engine)
print(orders.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
